# TerraPulse: NISAR surface change pipeline

Reading Earth's signals. Revealing its changes.

This notebook turns NISAR L-band radar time series (GCOV) into the map layers used by the TerraPulse web app, for two study sites:

- **Lower Indus, Sindh, Pakistan** (track 164, frame 76, 28 passes)
- **Amazon at Manaus, Brazil** (track 140, frame 90, 10 passes)

Steps:

1. **Rainfall**: daily NASA GPM IMERG (Late Run, V07) averaged over every zone of both sites.
2. **Search** the ASF archive and keep one product per date from a single processing version, plus the newest VALIDATED dates.
3. **Ingest**: stream only the study area out of each 6 GB HDF5 file (parallel range requests of the compressed chunks), drop samples flagged invalid by the NISAR mask, and average onto a common 100 m grid.
4. **Process**: open water calibrated on a reference water body and dry land, flooded vegetation, water occurrence, change metrics, surface rhythm classes, zone statistics.
5. **Download** one zip with the results.

**Before you start (one time):** IMERG lives at NASA GES DISC. In Earthdata Login (https://urs.earthdata.nasa.gov), open *Applications > Authorized Apps > Approve More Applications* and approve **NASA GESDISC DATA ARCHIVE**. Without it the rainfall step is skipped and the app falls back to NASA POWER.

**How to run:** open this notebook at https://colab.research.google.com (File > Upload notebook), then Runtime > Run all. When asked, paste your NASA Earthdata Login token (https://urs.earthdata.nasa.gov/profile, Generate Token). It is read with a hidden prompt and never saved in the notebook. If the runtime disconnects, run all again: finished work is skipped.

In [ ]:
# Choose what to run
RUN_IMERG = True          # daily GPM IMERG rainfall for both sites (about 10 to 20 minutes)
NISAR_SITES = ["amazon-manaus"]   # add "lower-indus" to re-run Sindh from scratch (about 20 minutes more)

In [ ]:
# Setup: work in a scratch folder and install the few libraries Colab lacks.
import os, sys, pathlib
IN_COLAB = "google.colab" in sys.modules
WORK = pathlib.Path("/content/terrapulse_run") if IN_COLAB else pathlib.Path.cwd() / "terrapulse_run"
(WORK / "terrapulse").mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
%pip install -q asf_search fsspec aiohttp h5py scipy pillow
print("Working in", WORK)

## Pipeline code

The cells below write the `terrapulse` Python package (the same code as the project repository).

In [ ]:
%%writefile terrapulse/__init__.py
"""TerraPulse: reading Earth's signals, revealing its changes.

Pipeline that turns NISAR L-band GCOV radar time series into web-ready
surface change layers (open water, flooded vegetation, change, rhythm).
"""
__version__ = "1.0.0"

In [ ]:
%%writefile terrapulse/config.py
"""Study sites and processing settings."""
from __future__ import annotations

from dataclasses import dataclass, field

# NISAR GCOV collections, in order of trust for a consistent time series.
CORE_COLLECTION = "NISAR_L2_GCOV_PROVISIONAL_V1"
VALIDATED_COLLECTION = "NISAR_L2_GCOV_V1"
BETA_COLLECTION = "NISAR_L2_GCOV_BETA_V1"

GCOV_GROUP = "/science/LSAR/GCOV/grids/frequencyA"
POLS = ("HHHH", "HVHV")

# Permanent NISAR instrument data gap (NISAR Data User Guide, availability page).
DATA_GAP = ("2026-07-27T22:03:25Z", "2026-08-10T00:55:27Z")


@dataclass(frozen=True)
class Zone:
    key: str
    name: str
    bbox: tuple[float, float, float, float]  # W, S, E, N (approximate)
    kind: str
    note: str = ""


@dataclass(frozen=True)
class Site:
    key: str
    name: str
    country: str
    bbox: tuple[float, float, float, float]  # W, S, E, N
    track: int
    frame: int
    direction: str
    start: str
    end: str
    res_m: int = 100
    zones: tuple[Zone, ...] = field(default_factory=tuple)
    water_ref: str = ""   # zone key of a permanent water body used to calibrate water thresholds
    dry_ref: str = ""     # zone key of land that is never flooded (negative reference)
    tagline: str = ""

    def zone(self, key: str) -> Zone | None:
        return next((z for z in self.zones if z.key == key), None)

    @property
    def wkt(self) -> str:
        w, s, e, n = self.bbox
        return f"POLYGON(({w} {s},{e} {s},{e} {n},{w} {n},{w} {s}))"


LOWER_INDUS = Site(
    key="lower-indus",
    name="Lower Indus, Sindh",
    country="Pakistan",
    bbox=(67.70, 24.30, 68.95, 25.50),
    track=164,
    frame=76,
    direction="DESCENDING",
    start="2025-11-01",
    end="2026-11-12",
    res_m=100,
    zones=(
        Zone("keenjhar", "Keenjhar Lake", (68.00, 24.85, 68.10, 25.05), "lake",
             "Freshwater reservoir that supplies Karachi; Ramsar site."),
        Zone("haleji", "Haleji Lake", (67.74, 24.77, 67.81, 24.84), "lake",
             "Wildlife sanctuary and Ramsar site."),
        Zone("badin-lagoons", "Badin coastal lagoons", (68.55, 24.30, 68.90, 24.55), "wetland",
             "Nurri and Jubho lagoons, Ramsar wetlands near the coast."),
        Zone("indus-floodplain", "Indus floodplain, Kotri to Thatta", (68.10, 24.85, 68.35, 25.35), "river",
             "River channel and kaccha (active floodplain) farmland."),
        Zone("left-bank-farms", "Left bank farmland, Tando Muhammad Khan to Matli", (68.45, 24.85, 68.80, 25.20), "cropland",
             "Irrigated rice, sugarcane and cotton belt."),
        Zone("hyderabad", "Hyderabad city", (68.30, 25.33, 68.42, 25.44), "urban",
             "Stable bright target used as a radiometric sanity check."),
        Zone("jhimpir-desert", "Rocky desert near Jhimpir (check area)", (67.85, 25.05, 67.95, 25.15), "desert",
             "Dry ground that is almost as dark as water in HH. Used to check that the water rule does not misfire."),
    ),
    water_ref="keenjhar",
    dry_ref="jhimpir-desert",
    tagline="Floods, wetlands and rice on the lower Indus",
)

# Zone boxes are approximate; they summarise areas, they do not define legal or ecological boundaries.
MANAUS = Site(
    key="amazon-manaus",
    name="Amazon at Manaus",
    country="Brazil",
    bbox=(-60.25, -3.45, -59.45, -2.88),
    track=140,
    frame=90,
    direction="DESCENDING",
    start="2026-06-01",
    end="2026-11-12",
    res_m=100,
    zones=(
        Zone("rio-negro", "Rio Negro at Manaus", (-60.22, -3.16, -60.06, -3.06), "river",
             "Wide black-water river channel, used as the permanent water reference."),
        Zone("meeting-waters", "Meeting of the Waters", (-59.92, -3.17, -59.80, -3.10), "river",
             "Where the dark Rio Negro meets the muddy Solimoes to form the Amazon."),
        Zone("careiro-varzea", "Careiro varzea floodplain", (-59.95, -3.40, -59.60, -3.18), "wetland",
             "Seasonally flooded forest, lakes and pasture on the Solimoes floodplain."),
        Zone("janauari", "Lago do Janauari area", (-60.10, -3.26, -59.98, -3.17), "wetland",
             "Floodplain lakes and flooded forest south of the Rio Negro."),
        Zone("amazon-east", "Amazon floodplain east", (-59.62, -3.35, -59.45, -3.05), "wetland",
             "Floodplain downstream of the confluence."),
        Zone("manaus-city", "Manaus city", (-60.06, -3.13, -59.93, -3.00), "urban",
             "City of about two million people."),
        Zone("ducke-forest", "Terra firme forest, Reserva Ducke (check area)", (-59.98, -2.98, -59.90, -2.92), "forest",
             "Upland rainforest that never floods. Used as the dry-land reference."),
    ),
    water_ref="rio-negro",
    dry_ref="ducke-forest",
    tagline="The flood pulse of the Amazon",
)

SITES = {LOWER_INDUS.key: LOWER_INDUS, MANAUS.key: MANAUS}

In [ ]:
%%writefile terrapulse/geo.py
"""Pure NumPy map projection helpers (WGS84 UTM and Web Mercator).

pyproj is not required, so the pipeline runs on machines where compiled
projection libraries are unavailable. UTM uses the Kruger series
(accuracy far below one metre inside a UTM zone).
"""
from __future__ import annotations

import numpy as np

A_WGS84 = 6378137.0
F_WGS84 = 1 / 298.257223563
K0 = 0.9996
E0 = 500000.0
R_MERC = 6378137.0

_n = F_WGS84 / (2 - F_WGS84)
_A = A_WGS84 / (1 + _n) * (1 + _n**2 / 4 + _n**4 / 64)
_ALPHA = (
    _n / 2 - 2 / 3 * _n**2 + 5 / 16 * _n**3,
    13 / 48 * _n**2 - 3 / 5 * _n**3,
    61 / 240 * _n**3,
)
_BETA = (
    _n / 2 - 2 / 3 * _n**2 + 37 / 96 * _n**3,
    1 / 48 * _n**2 + 1 / 15 * _n**3,
    17 / 480 * _n**3,
)
_DELTA = (
    2 * _n - 2 / 3 * _n**2 - 2 * _n**3,
    7 / 3 * _n**2 - 8 / 5 * _n**3,
    56 / 15 * _n**3,
)
_C = 2 * np.sqrt(_n) / (1 + _n)


def utm_zone_from_epsg(epsg: int) -> tuple[int, bool]:
    """Return (zone, northern_hemisphere) for EPSG 326xx / 327xx."""
    if 32601 <= epsg <= 32660:
        return epsg - 32600, True
    if 32701 <= epsg <= 32760:
        return epsg - 32700, False
    raise ValueError(f"EPSG {epsg} is not a WGS84 UTM zone")


def _central_meridian(zone: int) -> float:
    return np.radians(-183.0 + 6.0 * zone)


def lonlat_to_utm(lon, lat, epsg: int):
    """WGS84 lon/lat (degrees) to UTM easting/northing (metres)."""
    zone, north = utm_zone_from_epsg(epsg)
    lam = np.radians(np.asarray(lon, dtype=np.float64)) - _central_meridian(zone)
    phi = np.radians(np.asarray(lat, dtype=np.float64))
    sphi = np.sin(phi)
    t = np.sinh(np.arctanh(sphi) - _C * np.arctanh(_C * sphi))
    xi_p = np.arctan2(t, np.cos(lam))
    eta_p = np.arctanh(np.sin(lam) / np.sqrt(1 + t**2))
    e = eta_p.copy()
    n = xi_p.copy()
    for j, a in enumerate(_ALPHA, start=1):
        e += a * np.cos(2 * j * xi_p) * np.sinh(2 * j * eta_p)
        n += a * np.sin(2 * j * xi_p) * np.cosh(2 * j * eta_p)
    easting = E0 + K0 * _A * e
    northing = K0 * _A * n + (0.0 if north else 10000000.0)
    return easting, northing


def utm_to_lonlat(easting, northing, epsg: int):
    """UTM easting/northing (metres) to WGS84 lon/lat (degrees)."""
    zone, north = utm_zone_from_epsg(epsg)
    x = np.asarray(easting, dtype=np.float64)
    y = np.asarray(northing, dtype=np.float64) - (0.0 if north else 10000000.0)
    xi = y / (K0 * _A)
    eta = (x - E0) / (K0 * _A)
    xi_p = xi.copy()
    eta_p = eta.copy()
    for j, b in enumerate(_BETA, start=1):
        xi_p -= b * np.sin(2 * j * xi) * np.cosh(2 * j * eta)
        eta_p -= b * np.cos(2 * j * xi) * np.sinh(2 * j * eta)
    chi = np.arcsin(np.sin(xi_p) / np.cosh(eta_p))
    phi = chi.copy()
    for j, d in enumerate(_DELTA, start=1):
        phi += d * np.sin(2 * j * chi)
    lam = _central_meridian(zone) + np.arctan2(np.sinh(eta_p), np.cos(xi_p))
    return np.degrees(lam), np.degrees(phi)


def lonlat_to_merc(lon, lat):
    lon = np.asarray(lon, dtype=np.float64)
    lat = np.asarray(lat, dtype=np.float64)
    x = R_MERC * np.radians(lon)
    y = R_MERC * np.log(np.tan(np.pi / 4 + np.radians(lat) / 2))
    return x, y


def merc_to_lonlat(x, y):
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    lon = np.degrees(x / R_MERC)
    lat = np.degrees(2 * np.arctan(np.exp(y / R_MERC)) - np.pi / 2)
    return lon, lat


def bbox_to_utm_extent(bbox, epsg: int, densify: int = 50):
    """Densified lon/lat bbox (W, S, E, N) to UTM (xmin, ymin, xmax, ymax)."""
    w, s, e, n = bbox
    t = np.linspace(0, 1, densify)
    lon = np.concatenate([w + (e - w) * t, np.full(densify, e), e - (e - w) * t, np.full(densify, w)])
    lat = np.concatenate([np.full(densify, s), s + (n - s) * t, np.full(densify, n), n - (n - s) * t])
    x, y = lonlat_to_utm(lon, lat, epsg)
    return float(x.min()), float(y.min()), float(x.max()), float(y.max())


def self_test() -> None:
    """Round trip and a known reference point (Hyderabad, Sindh in zone 42N)."""
    lon, lat = np.array([68.37, 67.0, 69.9]), np.array([25.38, 24.0, 26.2])
    x, y = lonlat_to_utm(lon, lat, 32642)
    lon2, lat2 = utm_to_lonlat(x, y, 32642)
    assert np.allclose(lon, lon2, atol=1e-9) and np.allclose(lat, lat2, atol=1e-9)
    # Central meridian of zone 42 is 69E: easting must be exactly 500 km there.
    xc, _ = lonlat_to_utm(69.0, 25.0, 32642)
    assert abs(float(xc) - 500000.0) < 1e-6
    mx, my = lonlat_to_merc(lon, lat)
    lon3, lat3 = merc_to_lonlat(mx, my)
    assert np.allclose(lon, lon3) and np.allclose(lat, lat3)

In [ ]:
%%writefile terrapulse/remote.py
"""Fast, parallel reading of NISAR HDF5 chunks over HTTPS.

h5py reads chunks one by one, which is very slow over a high-latency link.
Here we read the chunk index once (through h5py on a cached remote file), then
download only the compressed chunks we need with merged, parallel HTTP range
requests, and decode them ourselves (deflate + shuffle, as used by NISAR GCOV).
"""
from __future__ import annotations

import os
import time
import zlib
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass
from pathlib import Path
from threading import Lock

import numpy as np
import requests

H5Z_DEFLATE, H5Z_SHUFFLE, H5Z_FLETCHER32 = 1, 2, 3


# --------------------------------------------------------------------------- auth
def get_token(env_file: str | os.PathLike | None = None) -> str:
    """Earthdata Login bearer token from EARTHDATA_TOKEN, a .env file, or a prompt."""
    tok = os.environ.get("EARTHDATA_TOKEN", "").strip()
    if tok:
        return tok
    candidates = [Path(env_file)] if env_file else [Path.cwd() / ".env", Path(__file__).resolve().parents[1] / ".env"]
    for p in candidates:
        if p.is_file():
            for line in p.read_text(encoding="utf8").splitlines():
                if line.strip().startswith("EARTHDATA_TOKEN="):
                    tok = line.split("=", 1)[1].strip().strip('"').strip("'")
                    if tok:
                        return tok
    from getpass import getpass

    return getpass("Earthdata Login token (input hidden): ").strip()


class SignedUrl:
    """Resolves the ASF URL to a short-lived CloudFront URL and refreshes it."""

    def __init__(self, url: str, token: str, max_age_s: float = 1500):
        self.url, self.token, self.max_age_s = url, token, max_age_s
        self._signed, self._t, self._lock = None, 0.0, Lock()

    def get(self, force: bool = False) -> str:
        with self._lock:
            if force or self._signed is None or time.time() - self._t > self.max_age_s:
                r = requests.get(self.url, headers={"Authorization": f"Bearer {self.token}", "Range": "bytes=0-0"},
                                 allow_redirects=False, timeout=60)
                if r.status_code in (301, 302, 303, 307, 308):
                    self._signed = r.headers["Location"]
                elif r.status_code in (200, 206):
                    self._signed = self.url  # served directly
                else:
                    raise RuntimeError(f"Earthdata auth failed ({r.status_code}). Check EARTHDATA_TOKEN.")
                self._t = time.time()
            return self._signed


def open_h5(url: str, token: str, block_size: int = 1 << 20):
    """Open a remote HDF5 file for metadata and chunk index access."""
    import aiohttp
    import fsspec
    import h5py

    fs = fsspec.filesystem("http", headers={"Authorization": f"Bearer {token}"},
                           client_kwargs={"timeout": aiohttp.ClientTimeout(total=900)})
    f = fs.open(url, "rb", block_size=block_size, cache_type="blockcache")
    return h5py.File(f, "r", rdcc_nbytes=64 << 20), f


# --------------------------------------------------------------------------- chunks
@dataclass
class ChunkRef:
    dataset: str
    row: int  # chunk start row (pixels)
    col: int  # chunk start col (pixels)
    offset: int
    size: int
    filter_mask: int


def filter_pipeline(dset) -> list[int]:
    plist = dset.id.get_create_plist()
    return [plist.get_filter(i)[0] for i in range(plist.get_nfilters())]


def chunk_index(dset, rows: tuple[int, int], cols: tuple[int, int]) -> list[ChunkRef]:
    """Chunks of `dset` intersecting the pixel window rows=[r0,r1), cols=[c0,c1)."""
    ch, cw = dset.chunks
    r0, r1 = rows
    c0, c1 = cols
    want = {(r, c) for r in range((r0 // ch) * ch, r1, ch) for c in range((c0 // cw) * cw, c1, cw)}
    refs: list[ChunkRef] = []
    name = dset.name
    if hasattr(dset.id, "chunk_iter"):
        def cb(info):
            key = (info.chunk_offset[0], info.chunk_offset[1])
            if key in want:
                refs.append(ChunkRef(name, key[0], key[1], info.byte_offset, info.size, info.filter_mask))
        dset.id.chunk_iter(cb)
    else:
        for (r, c) in want:
            info = dset.id.get_chunk_info_by_coord((r, c))
            if info.byte_offset is not None:
                refs.append(ChunkRef(name, r, c, info.byte_offset, info.size, info.filter_mask))
    return refs


def decode_chunk(raw: bytes, ref: ChunkRef, filters: list[int], chunk_shape, dtype) -> np.ndarray:
    data = raw
    # Filters are applied in order on write; undo them in reverse on read.
    for i in reversed(range(len(filters))):
        if ref.filter_mask & (1 << i):
            continue  # filter skipped for this chunk
        code = filters[i]
        if code == H5Z_DEFLATE:
            data = zlib.decompress(data)
        elif code == H5Z_SHUFFLE:
            itemsize = np.dtype(dtype).itemsize
            n = len(data) // itemsize
            b = np.frombuffer(data, dtype=np.uint8, count=n * itemsize)
            data = b.reshape(itemsize, n).T.tobytes() + data[n * itemsize:]
        elif code == H5Z_FLETCHER32:
            data = data[:-4]
        else:
            raise NotImplementedError(f"HDF5 filter {code} not supported")
    return np.frombuffer(data, dtype=dtype, count=int(np.prod(chunk_shape))).reshape(chunk_shape)


def _coalesce(refs: list[ChunkRef], max_gap: int, max_size: int) -> list[tuple[int, int, list[ChunkRef]]]:
    refs = sorted(refs, key=lambda r: r.offset)
    groups: list[tuple[int, int, list[ChunkRef]]] = []
    for r in refs:
        if groups:
            start, end, members = groups[-1]
            if r.offset - end <= max_gap and (r.offset + r.size - start) <= max_size:
                groups[-1] = (start, max(end, r.offset + r.size), members + [r])
                continue
        groups.append((r.offset, r.offset + r.size, [r]))
    return groups


def fetch_chunks(signed: SignedUrl, refs: list[ChunkRef], decode, workers: int = 8,
                 max_gap: int = 256 << 10, max_size: int = 8 << 20, retries: int = 5, log=print):
    """Download chunks with merged range requests; yields (ref, decoded_array)."""
    groups = _coalesce(refs, max_gap, max_size)
    total = sum(e - s for s, e, _ in groups)
    done_bytes, t0 = 0, time.time()
    session_local = {}

    def get_session():
        import threading
        tid = threading.get_ident()
        if tid not in session_local:
            session_local[tid] = requests.Session()
        return session_local[tid]

    def work(group):
        start, end, members = group
        for attempt in range(retries):
            try:
                r = get_session().get(signed.get(force=attempt > 1), headers={"Range": f"bytes={start}-{end - 1}"}, timeout=180)
                if r.status_code == 403:
                    signed.get(force=True)
                    raise RuntimeError("signed URL expired")
                r.raise_for_status()
                buf = r.content
                if len(buf) != end - start:
                    raise RuntimeError(f"short read {len(buf)} != {end - start}")
                return end - start, [(m, decode(buf[m.offset - start: m.offset - start + m.size], m)) for m in members]
            except Exception:  # network hiccup: back off and retry
                if attempt == retries - 1:
                    raise
                time.sleep(2 ** attempt)
        raise RuntimeError("unreachable")

    with ThreadPoolExecutor(workers) as ex:
        futures = [ex.submit(work, g) for g in groups]
        last = 0.0
        for fut in as_completed(futures):
            nbytes, items = fut.result()
            done_bytes += nbytes
            yield from items
            now = time.time()
            if now - last > 10 or done_bytes == total:
                rate = done_bytes / max(now - t0, 1e-6) / 1e6
                log(f"    {done_bytes / 1e6:7.1f} / {total / 1e6:.1f} MB  ({rate:.2f} MB/s)")
                last = now

In [ ]:
%%writefile terrapulse/catalog.py
"""Find NISAR GCOV granules for a site and pick one clean product per date."""
from __future__ import annotations

import json
from dataclasses import asdict, dataclass
from pathlib import Path

from .config import BETA_COLLECTION, CORE_COLLECTION, VALIDATED_COLLECTION, Site


@dataclass
class Granule:
    date: str
    start_time: str
    scene: str
    url: str
    collection: str
    crid: str
    mode: str
    pol: str
    bandwidth: str
    counter: int
    core: bool  # part of the single-CRID core time series

    @property
    def maturity(self) -> str:
        if self.collection == VALIDATED_COLLECTION:
            return "VALIDATED"
        if self.collection == BETA_COLLECTION:
            return "BETA"
        return "PROVISIONAL"


def _parse(scene: str) -> dict:
    # NISAR_L2_PR_GCOV_033_164_D_076_4005_DHDH_A_<start>_<stop>_P05036_N_F_J_001
    p = scene.split("_")
    return {"mode": p[8], "pol": p[9], "crid": p[13], "counter": int(p[-1])}


def search(site: Site) -> list[Granule]:
    import asf_search as asf

    results = asf.search(
        dataset="NISAR",
        processingLevel="GCOV",
        intersectsWith=site.wkt,
        relativeOrbit=site.track,
        frame=site.frame,
        flightDirection=site.direction,
        start=site.start,
        end=site.end,
        maxResults=1000,
    )
    out = []
    for g in results:
        pr = g.properties
        scene = pr.get("sceneName") or pr["fileID"]
        meta = _parse(scene)
        bw = pr.get("rangeBandwidth") or [""]
        out.append(Granule(
            date=pr["startTime"][:10],
            start_time=pr["startTime"],
            scene=scene,
            url=pr["url"],
            collection=pr.get("collectionName", ""),
            crid=meta["crid"],
            mode=meta["mode"],
            pol=meta["pol"],
            bandwidth=bw[0] if isinstance(bw, list) else str(bw),
            counter=meta["counter"],
            core=False,
        ))
    return out


def select(granules: list[Granule]) -> list[Granule]:
    """One product per date.

    Core series: the PROVISIONAL collection with its most common CRID, so the
    whole series comes from one processing version. VALIDATED products are kept
    (flagged core=False) because they postdate the PROVISIONAL series. BETA is dropped.
    """
    prov = [g for g in granules if g.collection == CORE_COLLECTION]
    crids = {}
    for g in prov:
        crids[g.crid] = crids.get(g.crid, 0) + 1
    core_crid = max(crids, key=crids.get) if crids else None

    by_date: dict[str, Granule] = {}
    for g in granules:
        if g.collection == BETA_COLLECTION:
            continue
        g.core = g.collection == CORE_COLLECTION and g.crid == core_crid
        cur = by_date.get(g.date)
        rank = (g.core or g.collection == VALIDATED_COLLECTION, g.collection == VALIDATED_COLLECTION, g.crid, g.counter)
        if cur is None:
            by_date[g.date] = g
        else:
            cur_rank = (cur.core or cur.collection == VALIDATED_COLLECTION, cur.collection == VALIDATED_COLLECTION, cur.crid, cur.counter)
            if rank > cur_rank:
                by_date[g.date] = g
    return [by_date[d] for d in sorted(by_date)]


def save(granules: list[Granule], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps([asdict(g) for g in granules], indent=1))


def load(path: Path) -> list[Granule]:
    return [Granule(**d) for d in json.loads(path.read_text())]

In [ ]:
%%writefile terrapulse/ingest.py
"""Stream one NISAR GCOV granule's AOI into the site's common 100 m UTM grid."""
from __future__ import annotations

import json
import math
import time
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np

from .catalog import Granule
from .config import GCOV_GROUP, POLS, Site
from .geo import bbox_to_utm_extent
from .remote import SignedUrl, chunk_index, decode_chunk, fetch_chunks, filter_pipeline, open_h5


@dataclass
class Grid:
    """North-up analysis grid in UTM. Cell (r, c) covers x0+c*res .. +res, y1-r*res .. -res."""
    epsg: int
    x0: float
    y1: float
    res: float
    width: int
    height: int

    @classmethod
    def for_site(cls, site: Site, epsg: int) -> Grid:
        xmin, ymin, xmax, ymax = bbox_to_utm_extent(site.bbox, epsg)
        r = float(site.res_m)
        x0, x1 = math.floor(xmin / r) * r, math.ceil(xmax / r) * r
        y0, y1 = math.floor(ymin / r) * r, math.ceil(ymax / r) * r
        return cls(epsg, x0, y1, r, int(round((x1 - x0) / r)), int(round((y1 - y0) / r)))

    def save(self, path: Path) -> None:
        path.write_text(json.dumps(asdict(self), indent=1))

    @classmethod
    def load(cls, path: Path) -> Grid:
        return cls(**json.loads(path.read_text()))


def _window(coords: np.ndarray, lo: float, hi: float) -> tuple[int, int]:
    """Index range [i0, i1) of pixel centres inside [lo, hi). Works for ascending or descending coords."""
    inside = np.nonzero((coords >= lo) & (coords < hi))[0]
    if inside.size == 0:
        return 0, 0
    return int(inside[0]), int(inside[-1]) + 1


def ingest_granule(g: Granule, site: Site, token: str, out_dir: Path, grid: Grid | None = None,
                   workers: int = 8, log=print) -> Path:
    out_dir.mkdir(parents=True, exist_ok=True)
    out = out_dir / f"{g.date}.npz"
    if out.exists():
        with np.load(out) as z:
            if "masked" in z.files:
                log(f"  {g.date}: cached")
                return out
        log(f"  {g.date}: cache predates the validity mask, rebuilding")

    t0 = time.time()
    h5, fobj = open_h5(g.url, token)
    try:
        grp = h5[GCOV_GROUP]
        epsg = int(grp["projection"].attrs["epsg_code"])
        x = grp["xCoordinates"][:]
        y = grp["yCoordinates"][:]
        spacing = float(abs(x[1] - x[0]))
        if grid is None:
            grid = Grid.for_site(site, epsg)
        if epsg != grid.epsg:
            raise RuntimeError(f"{g.date}: EPSG {epsg} differs from grid EPSG {grid.epsg}")

        gx1 = grid.x0 + grid.width * grid.res
        gy0 = grid.y1 - grid.height * grid.res
        c0, c1 = _window(x, grid.x0, gx1)
        r0, r1 = _window(y, gy0, grid.y1)
        if c1 <= c0 or r1 <= r0:
            raise RuntimeError(f"{g.date}: granule does not overlap the site grid")

        refs, meta = [], {}
        for pol in POLS:
            if pol not in grp:
                raise RuntimeError(f"{g.date}: {pol} missing in product")
            d = grp[pol]
            meta[d.name] = (pol, filter_pipeline(d), d.chunks, d.dtype, d.shape)
            refs += chunk_index(d, (r0, r1), (c0, c1))

        # Validity mask (NISAR GCOV): 0 = built from partially focused samples (swath gaps),
        # 1..n = subswath number (valid), 255 = outside the acquisition.
        mask_refs = []
        if "mask" in grp and grp["mask"].chunks == grp[POLS[0]].chunks:
            md = grp["mask"]
            meta[md.name] = ("mask", filter_pipeline(md), md.chunks, md.dtype, md.shape)
            mask_refs = chunk_index(md, (r0, r1), (c0, c1))
        else:
            log(f"  {g.date}: no usable mask layer, keeping all positive samples")
        log(f"  {g.date} {g.mode} {g.pol} {spacing:.0f} m: window rows {r0}-{r1} cols {c0}-{c1}, {len(refs)} chunks, index {time.time() - t0:.0f}s")

        # Verify our decoder against h5py on one chunk before trusting it.
        first = refs[0]
        pol, filters, cshape, dtype, shape = meta[first.dataset]
        signed = SignedUrl(g.url, token)
        sample = next(fetch_chunks(signed, [first], lambda raw, ref: decode_chunk(raw, ref, filters, cshape, dtype), workers=1, log=lambda *_: None))[1]
        rr, cc = min(first.row + cshape[0], shape[0]), min(first.col + cshape[1], shape[1])
        ref_vals = grp[pol][first.row:rr, first.col:cc]
        if not np.array_equal(np.nan_to_num(sample[: rr - first.row, : cc - first.col]), np.nan_to_num(ref_vals)):
            raise RuntimeError("chunk decoder self-check failed")
    finally:
        h5.close()
        fobj.close()

    H, W, res = grid.height, grid.width, grid.res
    sums = {p: np.zeros((H, W), np.float64) for p in POLS}
    cnts = {p: np.zeros((H, W), np.int32) for p in POLS}

    def decode(raw, ref):
        _, filters, cshape, dtype, _ = meta[ref.dataset]
        return decode_chunk(raw, ref, filters, cshape, dtype)

    masks = {}
    if mask_refs:
        for ref, arr in fetch_chunks(signed, mask_refs, decode, workers=workers, log=lambda *_: None):
            masks[(ref.row, ref.col)] = arr
    n_masked = 0

    for ref, arr in fetch_chunks(signed, refs, decode, workers=workers, log=log):
        pol, _, cshape, _, shape = meta[ref.dataset]
        rr = min(ref.row + cshape[0], shape[0])
        cc = min(ref.col + cshape[1], shape[1])
        a = arr[: rr - ref.row, : cc - ref.col]
        trow = np.floor((grid.y1 - y[ref.row:rr]) / res).astype(np.int64)
        tcol = np.floor((x[ref.col:cc] - grid.x0) / res).astype(np.int64)
        rin = (trow >= 0) & (trow < H)
        cin = (tcol >= 0) & (tcol < W)
        if not rin.any() or not cin.any():
            continue
        a = a[rin][:, cin]
        trow, tcol = trow[rin], tcol[cin]
        valid = np.isfinite(a) & (a > 0)
        mk = masks.get((ref.row, ref.col))
        if mk is not None:
            mk = mk[: rr - ref.row, : cc - ref.col][rin][:, cin]
            good = (mk >= 1) & (mk != 255)
            if pol == POLS[0]:
                n_masked += int((valid & ~good).sum())
            valid &= good
        if not valid.any():
            continue
        tr0, tc0 = trow.min(), tcol.min()
        nr, nc = trow.max() - tr0 + 1, tcol.max() - tc0 + 1
        idx = ((trow - tr0)[:, None] * nc + (tcol - tc0)[None, :])[valid]
        s = np.bincount(idx, weights=a[valid].astype(np.float64), minlength=nr * nc).reshape(nr, nc)
        n = np.bincount(idx, minlength=nr * nc).reshape(nr, nc)
        sums[pol][tr0:tr0 + nr, tc0:tc0 + nc] += s
        cnts[pol][tr0:tr0 + nr, tc0:tc0 + nc] += n

    full = (res / spacing) ** 2
    result = {}
    for p in POLS:
        with np.errstate(invalid="ignore", divide="ignore"):
            mean = (sums[p] / cnts[p]).astype(np.float32)
        mean[cnts[p] < 0.5 * full] = np.nan  # partial cells at swath edges
        result[p] = mean
    # Stored as dB in float16 (0.01 dB precision, far below radar noise) to keep the cache small.
    with np.errstate(invalid="ignore", divide="ignore"):
        hh_db = (10 * np.log10(result["HHHH"])).astype(np.float16)
        hv_db = (10 * np.log10(result["HVHV"])).astype(np.float16)
    np.savez_compressed(out, hh_db=hh_db, hv_db=hv_db,
                        looks=(cnts["HHHH"].astype(np.float32) / full).astype(np.float16),
                        masked=np.array(bool(mask_refs)))
    (out_dir / f"{g.date}.json").write_text(json.dumps({**asdict(g), "spacing_m": spacing, "epsg": epsg,
                                                       "mask_applied": bool(mask_refs), "masked_native_px": n_masked,
                                                       "seconds": round(time.time() - t0, 1)}, indent=1))
    log(f"  {g.date}: done in {time.time() - t0:.0f}s, valid {np.isfinite(result['HHHH']).mean() * 100:.0f}% of grid")
    return out

In [ ]:
%%writefile terrapulse/products.py
"""Turn the per-date 100 m backscatter stack into surface change products.

All thresholds are derived from the data (Otsu) and bounded to physically
sensible L-band ranges, and every choice is written to the manifest so the
web app can explain it.
"""
from __future__ import annotations

import json
from dataclasses import dataclass
from pathlib import Path

import numpy as np

from .config import Site
from .geo import utm_to_lonlat
from .ingest import Grid

# Class codes used in per-date maps.
NODATA, LAND, OPEN_WATER, FLOODED_VEG = 0, 1, 2, 3
# A pass is flagged as wind-roughened when less than this share of the reference lake reads as water.
WIND_LAKE_FRAC = 0.7


@dataclass
class Stack:
    dates: list[str]
    meta: list[dict]
    hh: np.ndarray  # [T, H, W] dB, NaN = no data
    hv: np.ndarray
    grid: Grid


def load_stack(root: Path) -> Stack:
    grid = Grid.load(root / "grid.json")
    files = sorted((root / "dates").glob("*.npz"))
    dates, meta, hh, hv = [], [], [], []
    for f in files:
        z = np.load(f)
        if "hh_db" in z:
            a, b = z["hh_db"].astype(np.float32), z["hv_db"].astype(np.float32)
        else:  # early cache format: linear power
            with np.errstate(invalid="ignore", divide="ignore"):
                a, b = 10 * np.log10(z["hh"]), 10 * np.log10(z["hv"])
        a[~np.isfinite(a)] = np.nan
        b[~np.isfinite(b)] = np.nan
        dates.append(f.stem)
        meta.append(json.loads(f.with_suffix(".json").read_text()))
        hh.append(a)
        hv.append(b)
    if not dates:
        raise RuntimeError(f"no ingested dates in {root / 'dates'}")
    return Stack(dates, meta, np.stack(hh), np.stack(hv), grid)


# --------------------------------------------------------------------------- helpers
def otsu(values: np.ndarray, lo: float, hi: float, bins: int = 256) -> float:
    v = values[np.isfinite(values)]
    v = v[(v >= lo) & (v <= hi)]
    hist, edges = np.histogram(v, bins=bins, range=(lo, hi))
    p = hist.astype(np.float64) / max(hist.sum(), 1)
    centers = (edges[:-1] + edges[1:]) / 2
    w0 = np.cumsum(p)
    mu = np.cumsum(p * centers)
    mu_t = mu[-1]
    with np.errstate(divide="ignore", invalid="ignore"):
        between = (mu_t * w0 - mu) ** 2 / (w0 * (1 - w0))
    between[~np.isfinite(between)] = 0
    return float(centers[int(np.argmax(between))])


def zone_mask(site: Site, grid: Grid, bbox) -> np.ndarray:
    """Boolean mask of grid cells whose centre falls inside a lon/lat bbox."""
    w, s, e, n = bbox
    xs = grid.x0 + (np.arange(grid.width) + 0.5) * grid.res
    ys = grid.y1 - (np.arange(grid.height) + 0.5) * grid.res
    X, Y = np.meshgrid(xs, ys)
    lon, lat = utm_to_lonlat(X, Y, grid.epsg)
    return (lon >= w) & (lon <= e) & (lat >= s) & (lat <= n)


def kmeans(x: np.ndarray, k: int, iters: int = 40, seed: int = 7) -> tuple[np.ndarray, np.ndarray]:
    """Plain NumPy k-means with k-means++ init. Returns (centroids, labels)."""
    rng = np.random.default_rng(seed)
    c = [x[rng.integers(len(x))]]
    for _ in range(1, k):
        d2 = np.min(((x[:, None, :] - np.array(c)[None]) ** 2).sum(-1), axis=1)
        c.append(x[rng.choice(len(x), p=d2 / d2.sum())])
    c = np.array(c)
    for _ in range(iters):
        lab = np.argmin(((x[:, None, :] - c[None]) ** 2).sum(-1), axis=1)
        new = np.array([x[lab == j].mean(0) if np.any(lab == j) else c[j] for j in range(k)])
        if np.allclose(new, c, atol=1e-5):
            break
        c = new
    return c, lab


RHYTHM_CLASSES = [
    # code, label, dance metaphor, colour
    (1, "Permanent water", "The still pool", "#1d4e89"),
    (2, "Seasonal water and wetland", "The tide", "#3fa7d6"),
    (3, "Flood-prone vegetation", "The swirl", "#7b5ea7"),
    (4, "Active cropland", "The harvest waltz", "#e0a526"),
    (5, "Woody vegetation, forest and orchards", "The slow sway", "#2e8b57"),
    (6, "Built-up area", "The steady beat", "#d1495b"),
    (7, "Bare soil and desert", "The quiet floor", "#c2b280"),
    (8, "Sparse vegetation and fallow", "The shuffle", "#9aa36b"),
]


def _rhythm_name(c) -> int:
    """Name a cluster centroid (in physical units) with simple radar rules."""
    hhm, hvm, hhs, hvs, occ, hvr, ffq = c
    if occ >= 0.6:
        return 1  # permanent water
    if occ >= 0.15:
        return 2  # seasonal water and wetland
    if ffq >= 0.12:
        return 3  # flood-prone vegetation
    if hhm > -4 and hhs < 2.0:
        return 6  # built-up: bright and steady
    if hvr >= 6 or hvs >= 2.0:
        return 4  # active cropland: big HV swing
    if hvm >= -16:
        return 5  # woody vegetation: high, steady HV
    if hvm < -24:
        return 7  # bare soil and desert
    return 8      # sparse vegetation and fallow


def rhythm_classes(X: np.ndarray, ok: np.ndarray, H: int, W: int, log=print, k: int = 10):
    """k-means on standardised yearly features, then rule-based names per cluster."""
    mu, sd = X.mean(0), X.std(0) + 1e-6
    rng = np.random.default_rng(3)
    idx = rng.choice(len(X), size=min(60000, len(X)), replace=False)
    cents, _ = kmeans((X[idx] - mu) / sd, k=k)
    lab = np.empty(len(X), np.int16)
    Z = (X - mu) / sd
    for s in range(0, len(Z), 200000):  # nearest centroid, in blocks to limit memory
        z = Z[s:s + 200000]
        lab[s:s + 200000] = np.argmin(((z[:, None, :] - cents[None]) ** 2).sum(-1), axis=1)
    cents_units = cents * sd + mu
    cluster_to_class = np.array([_rhythm_name(c) for c in cents_units], np.uint8)
    rhythm = np.zeros((H, W), np.uint8)
    rhythm[ok] = cluster_to_class[lab]
    log("  rhythm clusters -> classes: " + ", ".join(f"{i}:{c}" for i, c in enumerate(cluster_to_class)))
    return cents_units, cluster_to_class, rhythm


# --------------------------------------------------------------------------- main
def compute(stack: Stack, site: Site, log=print) -> dict:
    T, H, W = stack.hh.shape
    hh, hv = stack.hh.copy(), stack.hv.copy()
    valid = np.isfinite(hh) & np.isfinite(hv)
    modes = np.array([m.get("mode", "") for m in stack.meta])

    # 1. Mode bias check on temporally stable pixels (lowest 20% HH temporal std).
    with np.errstate(invalid="ignore"):
        std_hh0 = np.nanstd(hh, axis=0)
    stable = std_hh0 <= np.nanpercentile(std_hh0, 20)
    per_date_med = np.array([np.nanmedian(hh[t][stable]) for t in range(T)])
    per_date_med_hv = np.array([np.nanmedian(hv[t][stable]) for t in range(T)])
    bias = {}
    ref_mode = max(set(modes), key=list(modes).count)
    for m in sorted(set(modes)):
        sel = modes == m
        bias[m] = {"hh_db": float(np.nanmedian(per_date_med[sel]) - np.nanmedian(per_date_med[modes == ref_mode])),
                   "hv_db": float(np.nanmedian(per_date_med_hv[sel]) - np.nanmedian(per_date_med_hv[modes == ref_mode])),
                   "dates": int(sel.sum())}
    corrected = False
    if any(abs(b["hh_db"]) > 0.5 or abs(b["hv_db"]) > 0.5 for b in bias.values()):
        for t in range(T):
            hh[t] -= bias[modes[t]]["hh_db"]
            hv[t] -= bias[modes[t]]["hv_db"]
        corrected = True
    log(f"  mode bias (vs {ref_mode}): {json.dumps(bias)}; corrected={corrected}")

    with np.errstate(invalid="ignore"):
        hh_med = np.nanmedian(hh, axis=0)
        hv_med = np.nanmedian(hv, axis=0)
    d_hh = hh - hh_med[None]
    d_hv = hv - hv_med[None]

    # 2. Open water. At L-band, calm water sits near the noise floor in HV, but smooth dry
    #    desert is almost as dark in HH, so a blind histogram split misfires. We calibrate
    #    on a permanent reference lake (the darkest 40% of its zone by median HV) and take
    #    the 95th percentile of its values over all passes as thresholds.
    # Thresholds are calibrated separately per radar mode, because the 40 MHz mode has a
    # higher noise floor and reads dark targets about 1 dB brighter than the 20 MHz mode.
    ref = site.zone(site.water_ref) if site.water_ref else next((z for z in site.zones if z.kind == "lake"), None)
    core = None
    if ref is not None:
        zm = zone_mask(site, stack.grid, ref.bbox)
        core = zm & np.isfinite(hv_med) & (hv_med <= np.nanpercentile(hv_med[zm], 40))
        if core.sum() < 50:
            core = None
    dry_zone = site.zone(site.dry_ref) if site.dry_ref else next((z for z in site.zones if z.kind == "desert"), None)
    dry = zone_mask(site, stack.grid, dry_zone.bbox) if dry_zone else None
    thr, skill = {}, {}
    for m in sorted(set(modes)):
        sel = modes == m
        if core is not None and dry is not None and dry.sum() >= 50:
            # Pick the thresholds that best separate known water (reference lake) from
            # known dry land (desert check area): maximise Youden's J = hit rate - false alarms.
            L = np.stack([hh[sel][:, core].ravel(), hv[sel][:, core].ravel()], 1)
            D = np.stack([hh[sel][:, dry].ravel(), hv[sel][:, dry].ravel()], 1)
            L, D = L[np.all(np.isfinite(L), 1)], D[np.all(np.isfinite(D), 1)]
            best = (-1.0, -17.0, -24.0, 0.0, 0.0)
            for a in np.arange(-22, -13.9, 0.25):
                la, da = L[:, 0] < a, D[:, 0] < a
                for b in np.arange(-28, -19.9, 0.25):
                    tpr = float((la & (L[:, 1] < b)).mean())
                    fpr = float((da & (D[:, 1] < b)).mean())
                    if tpr - fpr > best[0]:
                        best = (tpr - fpr, float(a), float(b), tpr, fpr)
            thr[m] = (best[1], best[2])
            skill[m] = {"lake_hit_rate": round(best[3], 3), "desert_false_alarm": round(best[4], 3)}
        elif core is not None:
            thr[m] = (float(np.clip(np.nanpercentile(hh[sel][:, core], 95), -22, -14)),
                      float(np.clip(np.nanpercentile(hv[sel][:, core], 95), -28, -21)))
        else:
            v = valid[sel]
            thr[m] = (float(np.clip(otsu(hh[sel][v], -35, 5), -22, -14)), float(np.clip(otsu(hv[sel][v], -40, -5), -28, -21)))
    calib = (f"{ref.name} (water) vs {dry_zone.name} (dry land)" if skill else
             f"{ref.name}" if core is not None else "Otsu histogram split")
    if skill:
        log(f"  calibration skill per mode: {json.dumps(skill)}")
    t_hh_d = np.array([thr[m][0] for m in modes], np.float32)[:, None, None]
    t_hv_d = np.array([thr[m][1] for m in modes], np.float32)[:, None, None]
    steady_water = (hh < t_hh_d) & (hv < t_hv_d)
    # Newly flooded ground: both channels drop at least 3 dB below the pixel's own median
    # and end up close to water levels. Always-dark desert never changes, so it is excluded.
    new_water = (d_hh <= -3) & (d_hv <= -3) & (hh < t_hh_d + 2) & (hv < t_hv_d + 2)
    water = valid & (steady_water | new_water)
    t_hh, t_hv = thr[ref_mode]
    log(f"  open water thresholds per mode (calibrated on {calib}): " +
        ", ".join(f"{m}: HH < {a:.1f}, HV < {b:.1f} dB" for m, (a, b) in thr.items()))

    # Wind check: a permanent lake that stops looking like water has a wind-roughened surface.
    lake_frac = [None] * T
    if core is not None:
        for t in range(T):
            vv = valid[t] & core
            lake_frac[t] = round(float(water[t][vv].mean()), 3) if vv.sum() else None
        windy = [stack.dates[t] for t in range(T) if lake_frac[t] is not None and lake_frac[t] < WIND_LAKE_FRAC]
        log(f"  reference lake water fraction per pass: min {min(f for f in lake_frac if f is not None):.2f}; wind-roughened passes: {windy}")

    # 3. Flooded vegetation (L-band double bounce): HH jumps well above the pixel's
    #    own median, HH rises more than HV, and HH is bright.
    urban_like = hh_med > -4
    flooded = valid & ~water & (d_hh >= 3) & ((d_hh - d_hv) >= 2) & (hh >= -6) & ~urban_like[None]

    classes = np.full((T, H, W), NODATA, np.uint8)
    classes[valid] = LAND
    classes[water] = OPEN_WATER
    classes[flooded] = FLOODED_VEG
    # Light 3x3 majority-style cleanup of isolated pixels.
    try:
        from scipy.ndimage import median_filter
        for t in range(T):
            classes[t] = np.where(classes[t] == NODATA, NODATA, median_filter(classes[t], size=3))
    except ImportError:
        pass

    # 4. Temporal products.
    nvalid = valid.sum(0)
    with np.errstate(invalid="ignore", divide="ignore"):
        occurrence = np.where(nvalid > 0, (classes == OPEN_WATER).sum(0) / nvalid, np.nan)
        flood_freq = np.where(nvalid > 0, (classes == FLOODED_VEG).sum(0) / nvalid, np.nan)
        hv_range = np.nanpercentile(hv, 95, axis=0) - np.nanpercentile(hv, 5, axis=0)
        hh_std = np.nanstd(hh, axis=0)
        hv_std = np.nanstd(hv, axis=0)
        lin = 10 ** (hh / 10)
        cv = np.nanstd(lin, axis=0) / np.nanmean(lin, axis=0)
    enough = nvalid >= max(3, T // 3)

    # 5. Surface rhythm classes: k-means discovers groups, rules name them.
    feats = np.stack([hh_med, hv_med, hh_std, hv_std, occurrence, hv_range, flood_freq], -1)
    ok = enough & np.all(np.isfinite(feats), -1)
    X = feats[ok]
    if len(X) >= 1000:
        cents_units, cluster_to_class, rhythm = rhythm_classes(X, ok, H, W, log)
        # Small, distinctive classes can be absorbed by large clusters; pin them per pixel.
        rhythm[ok & (hh_med >= -5) & (hh_std <= 1.5)] = 6        # built-up: bright and steady HH
        rhythm[ok & (occurrence >= 0.8)] = 1                      # permanent water
    else:
        log("  rhythm skipped: not enough passes per pixel")
        cents_units, cluster_to_class, rhythm = np.zeros((0, 7)), np.zeros(0, np.uint8), np.zeros((H, W), np.uint8)

    # 6. Zone statistics per date.
    cell_km2 = (stack.grid.res / 1000) ** 2
    zones = {"all": ("Whole study area", np.ones((H, W), bool), "area")}
    for z in site.zones:
        zones[z.key] = (z.name, zone_mask(site, stack.grid, z.bbox), z.kind)
    zstats = {}
    for key, (zname, m, kind) in zones.items():
        rows = []
        zone_cells = max(int(m.sum()), 1)
        for t in range(T):
            v = valid[t] & m
            n = int(v.sum())
            if n < 0.3 * zone_cells:  # too little coverage to say anything
                rows.append(None)
                continue
            # Areas are scaled from the observed share to the whole zone, because passes
            # in the 20 MHz mode lose part of the area to masked swath gaps.
            scale = zone_cells / n * cell_km2
            w = float((classes[t][m] == OPEN_WATER).sum())
            f = float((classes[t][m] == FLOODED_VEG).sum())
            rows.append({
                "valid_frac": round(n / zone_cells, 3),
                "water_km2": round(w * scale, 2),
                "flooded_veg_km2": round(f * scale, 2),
                "water_km2_observed": round(w * cell_km2, 2),
                "hh_db": round(float(10 * np.log10(np.nanmean(10 ** (hh[t][v] / 10)))), 2),
                "hv_db": round(float(10 * np.log10(np.nanmean(10 ** (hv[t][v] / 10)))), 2),
            })
        zstats[key] = {"name": zname, "kind": kind, "area_km2": round(float(m.sum()) * cell_km2, 1), "series": rows}

    return {
        "hh": hh, "hv": hv, "valid": valid, "classes": classes,
        "occurrence": occurrence, "flood_freq": flood_freq, "hv_range": hv_range, "cv": cv,
        "hh_med": hh_med, "hv_med": hv_med, "rhythm": rhythm,
        "params": {
            "water_hh_db": round(t_hh, 2), "water_hv_db": round(t_hv, 2), "water_calibration": calib,
            "water_thresholds_by_mode": {m: {"hh_db": round(a, 2), "hv_db": round(b, 2), **skill.get(m, {})} for m, (a, b) in thr.items()},
            "wind_lake_frac": WIND_LAKE_FRAC,
            "reference_lake_water_frac": lake_frac,
            "water_rule": "HH and HV below thresholds calibrated on a permanent reference lake, or both channels at least 3 dB darker than the pixel's own median",
            "flood_rule": "HH rise >= 3 dB above pixel median, HH rise exceeds HV rise by >= 2 dB, HH >= -6 dB, not built-up",
            "mode_bias": bias, "mode_bias_corrected": corrected, "reference_mode": ref_mode,
            "kmeans_k": int(len(cents_units)), "rhythm_centroids": [[round(float(v), 3) for v in c] for c in cents_units],
            "rhythm_cluster_class": cluster_to_class.tolist(),
        },
        "zones": zstats,
    }

In [ ]:
%%writefile terrapulse/export.py
"""Write web-ready layers: Web Mercator 8-bit PNGs, a coarse time-series stack and a manifest."""
from __future__ import annotations

import json
import math
from datetime import date, timedelta
from pathlib import Path

import numpy as np
from PIL import Image

from .config import DATA_GAP, Site
from .geo import lonlat_to_merc, lonlat_to_utm, merc_to_lonlat
from .ingest import Grid
from .products import RHYTHM_CLASSES, Stack

# dB quantisation shared with the web app: value = 1 + round((dB - DB_MIN) / DB_STEP), 0 = no data.
DB_MIN, DB_MAX = -35.0, 5.0
DB_STEP = (DB_MAX - DB_MIN) / 254
# A pass with at least this much rain (area mean, mm) over the pass day and the two days before is flagged.
RAIN_AFFECTED_MM = 15.0
# Class PNG palette by code: 0 outside, 1 land (transparent), 2 open water, 3 flooded vegetation, 4 no radar data.
CLASS_PALETTE = ["#000000", "#00000000", "#1f6fd1", "#b04fd6", "#8a90a866"]


def quantize_db(a: np.ndarray) -> np.ndarray:
    q = np.zeros(a.shape, np.uint8)
    ok = np.isfinite(a)
    q[ok] = (1 + np.round((np.clip(a[ok], DB_MIN, DB_MAX) - DB_MIN) / DB_STEP)).astype(np.uint8)
    return q


class MercatorWarp:
    """Nearest-neighbour lookup from a Web Mercator output grid into the UTM analysis grid."""

    def __init__(self, site: Site, grid: Grid):
        w, s, e, n = site.bbox
        lat_c = (s + n) / 2
        self.res = grid.res / math.cos(math.radians(lat_c))  # mercator metres per pixel
        mx0, my0 = lonlat_to_merc(w, s)
        mx1, my1 = lonlat_to_merc(e, n)
        self.width = int(round((float(mx1) - float(mx0)) / self.res))
        self.height = int(round((float(my1) - float(my0)) / self.res))
        self.mx0, self.my1 = float(mx0), float(my1)
        mx = self.mx0 + (np.arange(self.width) + 0.5) * self.res
        my = self.my1 - (np.arange(self.height) + 0.5) * self.res
        MX, MY = np.meshgrid(mx, my)
        lon, lat = merc_to_lonlat(MX, MY)
        ux, uy = lonlat_to_utm(lon, lat, grid.epsg)
        c = np.floor((ux - grid.x0) / grid.res).astype(np.int64)
        r = np.floor((grid.y1 - uy) / grid.res).astype(np.int64)
        self.inside = (r >= 0) & (r < grid.height) & (c >= 0) & (c < grid.width)
        self.r, self.c = np.clip(r, 0, grid.height - 1), np.clip(c, 0, grid.width - 1)
        lon_w, lat_n = merc_to_lonlat(self.mx0, self.my1)
        lon_e, lat_s = merc_to_lonlat(self.mx0 + self.width * self.res, self.my1 - self.height * self.res)
        self.corners = [[float(lon_w), float(lat_n)], [float(lon_e), float(lat_n)],
                        [float(lon_e), float(lat_s)], [float(lon_w), float(lat_s)]]

    def __call__(self, a: np.ndarray, fill=0):
        out = a[self.r, self.c]
        out = np.where(self.inside, out, fill)
        return out.astype(a.dtype)


def save_png(a: np.ndarray, path: Path, palette: list[str] | None = None) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    im = Image.fromarray(a.astype(np.uint8), mode="L")
    if palette:
        # Palette entries are #RRGGBB or #RRGGBBAA; index 0 is always transparent (no data).
        im = Image.frombytes("P", im.size, a.astype(np.uint8).tobytes())
        pal, alpha = [0, 0, 0] * 256, [255] * 256
        for i, hexc in enumerate(palette):
            h = hexc.lstrip("#")
            pal[3 * i: 3 * i + 3] = [int(h[k:k + 2], 16) for k in (0, 2, 4)]
            alpha[i] = int(h[6:8], 16) if len(h) == 8 else 255
        alpha[0] = 0
        im.putpalette(pal)
        last = max(i for i in range(256) if i < len(palette)) + 1
        im.save(path, optimize=True, transparency=bytes(alpha[:last]))
        return
    im.save(path, optimize=True)


def _date_entry(d: str, m: dict, i: int, params: dict) -> dict:
    thr = params.get("water_thresholds_by_mode", {}).get(m.get("mode"), {})
    lake = (params.get("reference_lake_water_frac") or [None] * (i + 1))[i]
    return {
        "date": d, "maturity": "VALIDATED" if m.get("collection") == "NISAR_L2_GCOV_V1" else "PROVISIONAL",
        "core": m.get("core"), "crid": m.get("crid"), "mode": m.get("mode"), "pol": m.get("pol"),
        "spacing_m": m.get("spacing_m"), "scene": m.get("scene"),
        "thr_hh": thr.get("hh_db", params.get("water_hh_db")), "thr_hv": thr.get("hv_db", params.get("water_hv_db")),
        "lake_water_frac": lake, "wind_roughened": bool(lake is not None and lake < params.get("wind_lake_frac", 0.7)),
    }


def flag_rain_affected(entries: list[dict], daily: dict[str, float], threshold: float = RAIN_AFFECTED_MM) -> list[str]:
    """Add rain_3d_mm and rain_affected to each date entry, in place; return the flagged dates.

    Heavy rain in the three days before a pass (the pass day and the two days before) wets
    canopies and soil, which brightens HH and can mimic flooded vegetation, and roughens open
    water. Days missing from `daily` are skipped; with no rain data at all the pass is not flagged.
    """
    for e in entries:
        d0 = date.fromisoformat(e["date"])
        vals = [v for v in (daily.get(str(d0 - timedelta(days=k))) for k in range(3)) if v is not None]
        e["rain_3d_mm"] = round(sum(vals), 1) if vals else None
        e["rain_affected"] = bool(vals) and sum(vals) >= threshold
    return [e["date"] for e in entries if e["rain_affected"]]


def export(stack: Stack, res: dict, site: Site, out: Path, rain: dict | None = None,
           stack_factor: int = 4, log=print) -> Path:
    out.mkdir(parents=True, exist_ok=True)
    warp = MercatorWarp(site, stack.grid)
    T = len(stack.dates)
    log(f"  web grid: {warp.width} x {warp.height} px, {warp.res:.0f} m mercator")

    # Per-date backscatter and classes. Inside the study grid, pixels without valid radar data
    # (masked swath gaps) get class 4 so the map can show them as "no data" instead of dry land.
    for t, d in enumerate(stack.dates):
        save_png(warp(quantize_db(res["hh"][t])), out / "dates" / f"{d}_hh.png")
        save_png(warp(quantize_db(res["hv"][t])), out / "dates" / f"{d}_hv.png")
        cls = res["classes"][t].copy()
        cls[cls == 0] = 4
        save_png(warp(cls), out / "dates" / f"{d}_class.png",
                 palette=CLASS_PALETTE)

    def pct(a):
        q = np.zeros(a.shape, np.uint8)
        ok = np.isfinite(a)
        q[ok] = 1 + np.round(np.clip(a[ok], 0, 1) * 100).astype(np.uint8)
        return q

    def scaled(a, lo, hi):
        q = np.zeros(a.shape, np.uint8)
        ok = np.isfinite(a)
        q[ok] = 1 + np.round((np.clip(a[ok], lo, hi) - lo) / (hi - lo) * 254).astype(np.uint8)
        return q

    save_png(warp(pct(res["occurrence"])), out / "products" / "water_occurrence.png")
    save_png(warp(pct(res["flood_freq"])), out / "products" / "flooded_veg_frequency.png")
    save_png(warp(scaled(res["hv_range"], 0, 15)), out / "products" / "hv_range.png")
    save_png(warp(quantize_db(res["hh_med"])), out / "products" / "hh_median.png")
    save_png(warp(quantize_db(res["hv_med"])), out / "products" / "hv_median.png")
    save_png(warp(res["rhythm"]), out / "products" / "rhythm.png",
             palette=["#000000"] + [c[3] for c in RHYTHM_CLASSES])

    # Coarse stack for click-anywhere time series: uint8 [T][2][h][w].
    f = stack_factor
    hq = np.stack([warp(quantize_db(res["hh"][t])) for t in range(T)])
    vq = np.stack([warp(quantize_db(res["hv"][t])) for t in range(T)])
    h2, w2 = warp.height // f, warp.width // f

    def down(q):
        # Mean in dB of valid cells per block (0 = no data).
        v = q[:, : h2 * f, : w2 * f].reshape(T, h2, f, w2, f).astype(np.float32)
        m = v > 0
        s = (v * m).sum((2, 4))
        n = m.sum((2, 4))
        o = np.zeros((T, h2, w2), np.uint8)
        o[n > 0] = np.round(s[n > 0] / n[n > 0]).astype(np.uint8)
        return o

    series = np.stack([down(hq), down(vq)], axis=1)  # T,2,h,w
    (out / "stack.bin").write_bytes(series.tobytes())

    manifest = {
        "version": 1,
        "site": {"key": site.key, "name": site.name, "country": site.country, "bbox": site.bbox,
                 "track": site.track, "frame": site.frame, "direction": site.direction},
        "image": {"width": warp.width, "height": warp.height, "corners": warp.corners,
                  "res_m": site.res_m, "mercator_res_m": round(warp.res, 2)},
        "stack": {"file": "stack.bin", "factor": f, "width": w2, "height": h2, "order": "T,pol(hh,hv),y,x"},
        "db_quant": {"min": DB_MIN, "max": DB_MAX, "step": DB_STEP, "nodata": 0},
        "dates": [_date_entry(d, m, i, res["params"]) for i, (d, m) in enumerate(zip(stack.dates, stack.meta, strict=True))],
        "data_gap": DATA_GAP,
        "params": res["params"],
        "classes": {"0": "outside study area", "1": "land", "2": "open water", "3": "flooded vegetation (likely)",
                    "4": "no valid radar data (masked swath gap)"},
        "rhythm_classes": [{"code": c[0], "label": c[1], "dance": c[2], "color": c[3]} for c in RHYTHM_CLASSES],
        "zones": res["zones"],
        "zone_boxes": {z.key: {"name": z.name, "bbox": z.bbox, "kind": z.kind, "note": z.note} for z in site.zones},
        "rain": rain,
    }
    daily = ((rain or {}).get("series") or {}).get("all") or {}
    if daily:
        flagged = flag_rain_affected(manifest["dates"], daily)
        manifest["params"]["rain_affected_mm_3d"] = RAIN_AFFECTED_MM
        log(f"  rain-affected passes (>= {RAIN_AFFECTED_MM} mm in 3 days): {flagged}")
    manifest["site"]["tagline"] = site.tagline
    manifest["site"]["water_ref"] = site.water_ref
    manifest["site"]["dry_ref"] = site.dry_ref
    (out / "manifest.json").write_text(json.dumps(manifest, indent=1))
    _update_index(out.parent, site, manifest)
    from .stac import build as build_stac
    build_stac(out.parent, log=log)
    size = sum(p.stat().st_size for p in out.rglob("*") if p.is_file()) / 1e6
    log(f"  wrote {out} ({size:.1f} MB)")
    return out


def _update_index(root: Path, site: Site, manifest: dict) -> None:
    """Keep web/data/sites.json listing every exported site, for the app's site switcher."""
    path = root / "sites.json"
    index = json.loads(path.read_text()) if path.exists() else {"sites": []}
    entry = {"key": site.key, "name": site.name, "country": site.country, "tagline": site.tagline,
             "bbox": site.bbox, "passes": len(manifest["dates"]),
             "first": manifest["dates"][0]["date"], "last": manifest["dates"][-1]["date"]}
    index["sites"] = [s for s in index["sites"] if s["key"] != site.key] + [entry]
    order = {"lower-indus": 0}
    index["sites"].sort(key=lambda s: (order.get(s["key"], 1), s["key"]))
    path.write_text(json.dumps(index, indent=1))

In [ ]:
%%writefile terrapulse/rain.py
"""Daily rainfall context from NASA POWER (MERRA-2 / GEOS-IT based, no login needed)."""
from __future__ import annotations

import json
import urllib.request

from .config import Site

POWER_URL = ("https://power.larc.nasa.gov/api/temporal/daily/point?parameters=PRECTOTCORR&community=AG"
             "&longitude={lon:.4f}&latitude={lat:.4f}&start={start}&end={end}&format=JSON")


def _point(lon: float, lat: float, start: str, end: str) -> dict[str, float]:
    url = POWER_URL.format(lon=lon, lat=lat, start=start.replace("-", ""), end=end.replace("-", ""))
    req = urllib.request.Request(url, headers={"User-Agent": "terrapulse"})
    with urllib.request.urlopen(req, timeout=120) as r:
        data = json.load(r)
    p = data["properties"]["parameter"]["PRECTOTCORR"]
    return {f"{k[:4]}-{k[4:6]}-{k[6:]}": round(v, 2) for k, v in p.items() if v is not None and v >= 0}


def fetch(site: Site, log=print) -> dict:
    points = {"all": ((site.bbox[0] + site.bbox[2]) / 2, (site.bbox[1] + site.bbox[3]) / 2)}
    for z in site.zones:
        points[z.key] = ((z.bbox[0] + z.bbox[2]) / 2, (z.bbox[1] + z.bbox[3]) / 2)
    series = {}
    for key, (lon, lat) in points.items():
        try:
            series[key] = _point(lon, lat, site.start, site.end)
        except Exception as e:
            log(f"  rain {key}: {type(e).__name__}: {e}")
    log(f"  rainfall: {len(series)} points from NASA POWER")
    return {
        "source": "NASA POWER daily PRECTOTCORR (MERRA-2 and GEOS-IT), mm/day",
        "caveat": "Model reanalysis at about 50 km resolution. It smooths out local convective downpours, "
                  "so single storm totals can be much higher on the ground than shown.",
        "series": series,
    }

In [ ]:
%%writefile terrapulse/rain_imerg.py
"""Daily rainfall from NASA GPM IMERG (Late Run, V07, 0.1 degree), averaged over each zone.

IMERG merges microwave and infrared satellite estimates, so unlike a reanalysis model it
captures local monsoon downpours. Each daily file is opened remotely and only the small
window around the study site is read.

Needs an Earthdata Login token whose account has approved the "NASA GESDISC DATA ARCHIVE"
application (Earthdata Login profile > Applications > Authorized Apps > Approve More Applications).
"""
from __future__ import annotations

import json
import re
import time
import urllib.request
from pathlib import Path

import numpy as np

from .config import Site
from .remote import open_h5

CMR = "https://cmr.earthdata.nasa.gov/search/granules.json"
SHORT_NAME, VERSION = "GPM_3IMERGDL", "07"
SOURCE = "NASA GPM IMERG Late Run daily, V07, 0.1 degree (GES DISC)"


def granules(start: str, end: str) -> dict[str, str]:
    """Map date -> HTTPS URL for every daily IMERG Late file between start and end."""
    out: dict[str, str] = {}
    page = 1
    while True:
        url = (f"{CMR}?short_name={SHORT_NAME}&version={VERSION}&temporal={start}T00:00:00Z,{end}T23:59:59Z"
               f"&page_size=2000&page_num={page}&sort_key=start_date")
        with urllib.request.urlopen(urllib.request.Request(url, headers={"User-Agent": "terrapulse"}), timeout=120) as r:
            entries = json.load(r)["feed"]["entry"]
        for e in entries:
            m = re.search(r"3IMERG\.(\d{8})", e["title"])
            href = next((lk["href"] for lk in e.get("links", []) if lk["href"].startswith("https://") and lk["href"].endswith(".nc4")
                         and "opendap" not in lk["href"]), None)
            if m and href:
                d = m.group(1)
                out[f"{d[:4]}-{d[4:6]}-{d[6:]}"] = href  # later versions in the listing overwrite earlier ones
        if len(entries) < 2000:
            return dict(sorted(out.items()))
        page += 1


def _find_precip(h5):
    names = []
    h5.visititems(lambda n, o: names.append(n) if getattr(o, "ndim", 0) >= 2 else None)
    for want in ("precipitation", "precipitationCal"):
        for n in names:
            if n.split("/")[-1] == want:
                return h5[n]
    raise KeyError(f"no precipitation variable among {names}")


def _coord(h5, name: str):
    for n in (name, f"Grid/{name}"):
        if n in h5:
            return h5[n][:]
    raise KeyError(name)


def _site_window(site: Site, lon: np.ndarray, lat: np.ndarray, pad: float = 0.1):
    w, s, e, n = site.bbox
    i = np.nonzero((lon >= w - pad) & (lon <= e + pad))[0]
    j = np.nonzero((lat >= s - pad) & (lat <= n + pad))[0]
    return int(i[0]), int(i[-1]) + 1, int(j[0]), int(j[-1]) + 1


def _zone_means(site: Site, lon_w: np.ndarray, lat_w: np.ndarray, P: np.ndarray) -> dict[str, float]:
    """P is [lon, lat] in mm/day. Mean over grid cells whose centres fall in each zone (nearest cell if none)."""
    out = {}
    boxes = {"all": site.bbox, **{z.key: z.bbox for z in site.zones}}
    LON, LAT = np.meshgrid(lon_w, lat_w, indexing="ij")
    for key, (w, s, e, n) in boxes.items():
        m = (LON >= w) & (LON <= e) & (LAT >= s) & (LAT <= n) & np.isfinite(P)
        if not m.any():
            ci, cj = np.argmin(np.abs(lon_w - (w + e) / 2)), np.argmin(np.abs(lat_w - (s + n) / 2))
            v = P[ci, cj]
        else:
            v = float(np.nanmean(P[m]))
        if np.isfinite(v):
            out[key] = round(float(v), 2)
    return out


def fetch(sites: list[Site], token: str, cache_root: Path, log=print) -> dict[str, Path]:
    """Read IMERG for all sites in one pass over the daily files. Resumable per site."""
    start = min(s.start for s in sites)
    end = max(s.end for s in sites)
    urls = granules(start, end)
    log(f"IMERG: {len(urls)} daily files {start} to {end}")
    paths, results = {}, {}
    for s in sites:
        p = cache_root / s.key / "rain_imerg.json"
        p.parent.mkdir(parents=True, exist_ok=True)
        paths[s.key] = p
        results[s.key] = json.loads(p.read_text())["series"] if p.exists() else {}
    t0, done = time.time(), 0
    for k, (date, url) in enumerate(urls.items(), 1):
        todo = [s for s in sites if s.start <= date <= s.end and date not in results[s.key].get("all", {})]
        if not todo:
            continue
        for attempt in range(4):
            try:
                h5, f = open_h5(url, token, block_size=256 * 1024)
                try:
                    lon, lat = _coord(h5, "lon"), _coord(h5, "lat")
                    var = _find_precip(h5)
                    units = str(var.attrs.get("units", b"mm/day"), "utf8") if isinstance(var.attrs.get("units"), bytes) else str(var.attrs.get("units", "mm/day"))
                    for s in todo:
                        i0, i1, j0, j1 = _site_window(s, lon, lat)
                        # Daily files are (time, lon, lat); be robust to (time, lat, lon).
                        if var.shape[-2] == lon.size:
                            P = np.asarray(var[0, i0:i1, j0:j1], np.float64)
                        else:
                            P = np.asarray(var[0, j0:j1, i0:i1], np.float64).T
                        P[P < 0] = np.nan
                        if "hr" in units:
                            P *= 24.0
                        for zone, mm in _zone_means(s, lon[i0:i1], lat[j0:j1], P).items():
                            results[s.key].setdefault(zone, {})[date] = mm
                finally:
                    h5.close()
                    f.close()
                break
            except Exception as e:
                msg = str(e)
                if "403" in msg or "401" in msg:
                    raise RuntimeError("GES DISC refused the download. In Earthdata Login, open Applications > "
                                       "Authorized Apps > Approve More Applications and approve 'NASA GESDISC DATA ARCHIVE', "
                                       "then run this step again.") from e
                if attempt == 3:
                    log(f"  {date}: skipped ({type(e).__name__}: {msg[:80]})")
                time.sleep(2 ** attempt)
        done += 1
        if done % 20 == 0:
            for s in sites:
                _save(paths[s.key], results[s.key])
            log(f"  {k}/{len(urls)} files, {time.time() - t0:.0f}s")
    for s in sites:
        _save(paths[s.key], results[s.key])
    log(f"IMERG done in {time.time() - t0:.0f}s")
    return paths


def _save(path: Path, series: dict) -> None:
    path.write_text(json.dumps({
        "source": SOURCE,
        "caveat": "Satellite estimate on a 0.1 degree (about 11 km) grid, averaged over each area. "
                  "Late Run products are calibrated less than the Final Run, which is not yet available for 2026.",
        "series": series,
    }))


def load(path: Path) -> dict | None:
    return json.loads(path.read_text()) if path.exists() else None

In [ ]:
%%writefile terrapulse/stac.py
"""Publish the web layers as a static STAC catalog (SpatioTemporal Asset Catalog, v1.1.0).

Every exported pass becomes a STAC Item whose assets are the 8-bit PNG layers the web app
uses. The raster extension records how to turn stored pixel values back into physical units
(dB, fractions), the projection extension georeferences the images in Web Mercator, and the
classification extension names the class codes. Tools such as STAC Browser, pystac or QGIS
can then discover and use TerraPulse outputs without reading any of our code.

    python -m terrapulse stac        # rebuild web/data/catalog.json from the exported manifests
"""
from __future__ import annotations

import json
import shutil
from pathlib import Path
from urllib.parse import quote

from .export import CLASS_PALETTE, DB_MIN, DB_STEP
from .geo import lonlat_to_merc

STAC_VERSION = "1.1.0"
EXT = {
    "sar": "https://stac-extensions.github.io/sar/v1.0.0/schema.json",
    "sat": "https://stac-extensions.github.io/sat/v1.0.0/schema.json",
    "proj": "https://stac-extensions.github.io/projection/v2.0.0/schema.json",
    "raster": "https://stac-extensions.github.io/raster/v1.1.0/schema.json",
    "class": "https://stac-extensions.github.io/classification/v1.1.0/schema.json",
}
REPO = "https://github.com/sudais-khalid/terrapulse"
APP = "https://sudais-khalid.github.io/terrapulse/"
ASF_SEARCH = "https://api.daac.asf.alaska.edu/services/search/param?output=geojson&granule_list="
NISAR_L_BAND_GHZ = 1.257
PNG = "image/png"

# Stored value q (1..255, 0 = no data) to physical value: q * scale + offset.
DB_BAND = {"nodata": 0, "data_type": "uint8", "unit": "dB", "scale": DB_STEP, "offset": DB_MIN - DB_STEP}
FRACTION_BAND = {"nodata": 0, "data_type": "uint8", "unit": "1", "scale": 0.01, "offset": -0.01}
HV_RANGE_BAND = {"nodata": 0, "data_type": "uint8", "unit": "dB", "scale": 15 / 254, "offset": -15 / 254}


def _start_time(scene: str, day: str) -> str:
    # NISAR_L2_PR_GCOV_..._A_20260623T221633_20260623T221711_P05023_...
    for part in scene.split("_"):
        if len(part) == 15 and part[8] == "T" and part[:8].isdigit():
            return f"{part[:4]}-{part[4:6]}-{part[6:8]}T{part[9:11]}:{part[11:13]}:{part[13:15]}Z"
    return f"{day}T00:00:00Z"


def _classes(items: list[tuple[int, str, str, str]]) -> list[dict]:
    out = []
    for value, name, description, color in items:
        c = {"value": value, "name": name, "description": description}
        if color and len(color.lstrip("#")) >= 6:
            c["color_hint"] = color.lstrip("#")[:6].upper()
        out.append(c)
    return out


def _slug(text: str) -> str:
    return "-".join("".join(ch if ch.isalnum() else " " for ch in text.lower()).split())


def _geometry(m: dict) -> tuple[dict, list[float], dict]:
    corners = m["image"]["corners"]  # NW, NE, SE, SW in lon/lat
    lons, lats = [c[0] for c in corners], [c[1] for c in corners]
    geom = {"type": "Polygon", "coordinates": [[*corners, corners[0]]]}
    w, h = m["image"]["width"], m["image"]["height"]
    # Pixel size from the exact corners (the manifest's mercator_res_m is rounded for display).
    mx0, my1 = (float(v) for v in lonlat_to_merc(corners[0][0], corners[0][1]))
    mx1, my0 = (float(v) for v in lonlat_to_merc(corners[2][0], corners[2][1]))
    rx, ry = (mx1 - mx0) / w, (my1 - my0) / h
    proj = {"proj:code": "EPSG:3857", "proj:shape": [h, w], "proj:transform": [rx, 0.0, mx0, 0.0, -ry, my1]}
    return geom, [min(lons), min(lats), max(lons), max(lats)], proj


def _item(m: dict, d: dict, geom: dict, bbox: list[float], proj: dict, class_list: list[dict]) -> dict:
    site, day = m["site"], d["date"]
    pols = ["HH", "HV"]
    props = {
        "datetime": _start_time(d.get("scene") or "", day),
        "title": f"{site['name']}, NISAR pass {day}",
        "platform": "nisar", "instruments": ["l-sar"], "gsd": float(m["image"]["res_m"]),
        "sar:instrument_mode": str(d.get("mode") or "unknown"), "sar:frequency_band": "L",
        "sar:center_frequency": NISAR_L_BAND_GHZ, "sar:polarizations": pols, "sar:product_type": "GCOV",
        "sat:orbit_state": site["direction"].lower(), "sat:relative_orbit": site["track"],
        **proj,
        "terrapulse:maturity": d.get("maturity"), "terrapulse:crid": d.get("crid"),
        "terrapulse:core_series": bool(d.get("core")), "terrapulse:source_granule": d.get("scene"),
        "terrapulse:water_threshold_hh_db": d.get("thr_hh"), "terrapulse:water_threshold_hv_db": d.get("thr_hv"),
        "terrapulse:wind_roughened": d.get("wind_roughened"),
        "terrapulse:rain_3d_mm": d.get("rain_3d_mm"), "terrapulse:rain_affected": d.get("rain_affected"),
    }
    links = [{"rel": "root", "href": "../../catalog.json", "type": "application/json"},
             {"rel": "parent", "href": "../collection.json", "type": "application/json"},
             {"rel": "collection", "href": "../collection.json", "type": "application/json"}]
    if d.get("scene"):
        links.append({"rel": "derived_from", "href": ASF_SEARCH + quote(d["scene"]), "type": "application/geo+json",
                      "title": "Source NISAR L2 GCOV granule (ASF search)"})
    links.append({"rel": "alternate", "href": f"{APP}#site={site['key']}&mode=explore&layer=water&d={day}", "type": "text/html",
                  "title": "Open this pass in TerraPulse"})
    assets = {
        "hh": {"href": f"../dates/{day}_hh.png", "type": PNG, "title": "HH backscatter (gamma-naught, dB)",
               "roles": ["data"], "raster:bands": [DB_BAND]},
        "hv": {"href": f"../dates/{day}_hv.png", "type": PNG, "title": "HV backscatter (gamma-naught, dB)",
               "roles": ["data"], "raster:bands": [DB_BAND]},
        "class": {"href": f"../dates/{day}_class.png", "type": PNG, "title": "Open water and likely flooded vegetation",
                  "roles": ["data", "visual"], "raster:bands": [{"nodata": 0, "data_type": "uint8"}],
                  "classification:classes": class_list},
    }
    return {"type": "Feature", "stac_version": STAC_VERSION,
            "stac_extensions": [EXT["sar"], EXT["sat"], EXT["proj"], EXT["raster"], EXT["class"]],
            "id": f"{site['key']}-{day}", "collection": site["key"], "geometry": geom, "bbox": bbox,
            "properties": props, "links": links, "assets": assets}


def build_site(root: Path) -> dict:
    """Write <site>/collection.json and <site>/items/*.json; return the collection."""
    m = json.loads((root / "manifest.json").read_text())
    site = m["site"]
    geom, bbox, proj = _geometry(m)
    class_list = _classes([(int(k), _slug(v), v, CLASS_PALETTE[int(k)] if int(k) < len(CLASS_PALETTE) else "")
                           for k, v in m["classes"].items() if k != "0"])
    rhythm = _classes([(c["code"], _slug(c["label"]), f"{c['label']} ({c['dance']})", c["color"])
                       for c in m["rhythm_classes"]])
    items_dir = root / "items"
    if items_dir.exists():
        shutil.rmtree(items_dir)
    items_dir.mkdir(parents=True)
    times = []
    for d in m["dates"]:
        item = _item(m, d, geom, bbox, proj, class_list)
        times.append(item["properties"]["datetime"])
        (items_dir / f"{d['date']}.json").write_text(json.dumps(item, indent=1))

    def product(name, title, band=None, classes=None):
        a = {"href": f"products/{name}.png", "type": PNG, "title": title, "roles": ["data"], **proj}
        a["raster:bands"] = [band or {"nodata": 0, "data_type": "uint8"}]
        if classes:
            a["classification:classes"] = classes
        return a

    params = m["params"]
    collection = {
        "type": "Collection", "stac_version": STAC_VERSION,
        "stac_extensions": [EXT["sar"], EXT["sat"], EXT["proj"], EXT["raster"], EXT["class"]],
        "id": site["key"], "title": f"TerraPulse: {site['name']}, {site['country']}",
        "description": (f"{site.get('tagline', '')}. Surface change layers from {len(m['dates'])} NISAR L2 GCOV passes "
                        f"(track {site['track']}, frame {site['frame']}, {site['direction'].lower()}), resampled to "
                        f"{m['image']['res_m']} m and published as 8-bit PNGs in Web Mercator. Open water uses HH and HV "
                        f"thresholds calibrated per radar mode on a permanent water body ({site.get('water_ref')}) against "
                        f"never-flooded land ({site.get('dry_ref')}). Passes are flagged when wind roughened the reference "
                        f"water or when at least {params.get('rain_affected_mm_3d', 15)} mm of GPM IMERG rain fell in "
                        "the three days before."),
        "keywords": ["NISAR", "SAR", "L-band", "GCOV", "open water", "flooded vegetation", "change detection", site["country"]],
        "license": "Apache-2.0",
        "providers": [
            {"name": "NASA-ISRO SAR mission (NASA JPL, ISRO)", "roles": ["producer", "licensor"], "url": "https://nisar.jpl.nasa.gov/"},
            {"name": "Alaska Satellite Facility DAAC", "roles": ["host"], "url": "https://asf.alaska.edu/"},
            {"name": "TerraPulse (Sudais Khalid, Sunnia Aslam)", "roles": ["processor", "host"], "url": REPO},
        ],
        "extent": {"spatial": {"bbox": [bbox]}, "temporal": {"interval": [[min(times), max(times)]]}},
        "summaries": {
            "platform": ["nisar"], "instruments": ["l-sar"], "gsd": [float(m["image"]["res_m"])],
            "sar:frequency_band": ["L"], "sar:product_type": ["GCOV"], "sar:polarizations": ["HH", "HV"],
            "sar:instrument_mode": sorted({str(d.get("mode")) for d in m["dates"]}),
            "sat:orbit_state": [site["direction"].lower()], "sat:relative_orbit": [site["track"]],
            "terrapulse:maturity": sorted({d.get("maturity") for d in m["dates"]}),
        },
        "item_assets": {
            "hh": {"type": PNG, "title": "HH backscatter (gamma-naught, dB)", "roles": ["data"], "raster:bands": [DB_BAND]},
            "hv": {"type": PNG, "title": "HV backscatter (gamma-naught, dB)", "roles": ["data"], "raster:bands": [DB_BAND]},
            "class": {"type": PNG, "title": "Open water and likely flooded vegetation", "roles": ["data", "visual"],
                      "classification:classes": class_list},
        },
        "assets": {
            "water_occurrence": product("water_occurrence", "Share of passes with open water", FRACTION_BAND),
            "flooded_veg_frequency": product("flooded_veg_frequency", "Share of passes with likely flooded vegetation",
                                             FRACTION_BAND),
            "hv_range": product("hv_range", "Vegetation swing: HV 95th minus 5th percentile (dB)", HV_RANGE_BAND),
            "hh_median": product("hh_median", "Median HH backscatter over the year (dB)", DB_BAND),
            "hv_median": product("hv_median", "Median HV backscatter over the year (dB)", DB_BAND),
            "rhythm": product("rhythm", "Surface rhythm: places grouped by how their radar signal moved through the year",
                              classes=rhythm),
            "manifest": {"href": "manifest.json", "type": "application/json", "roles": ["metadata"],
                         "title": "TerraPulse manifest: calibration, quality flags, zone statistics, rainfall"},
            "stack": {"href": "stack.bin", "type": "application/octet-stream", "roles": ["data"],
                      "title": "Coarse HH/HV time series cube, uint8 [time, pol, y, x], same dB encoding",
                      "raster:bands": [DB_BAND]},
        },
        "links": [{"rel": "root", "href": "../catalog.json", "type": "application/json"},
                  {"rel": "parent", "href": "../catalog.json", "type": "application/json"},
                  {"rel": "license", "href": f"{REPO}/blob/main/LICENSE", "type": "text/html"},
                  {"rel": "about", "href": f"{APP}#site={site['key']}", "type": "text/html", "title": "Open in TerraPulse"},
                  *({"rel": "item", "href": f"items/{d['date']}.json", "type": "application/geo+json"} for d in m["dates"])],
    }
    (root / "collection.json").write_text(json.dumps(collection, indent=1))
    return collection


def build(web_data: Path, log=print) -> Path:
    """Rebuild the whole catalog from every site listed in web_data/sites.json."""
    index = json.loads((web_data / "sites.json").read_text())
    links = [{"rel": "root", "href": "./catalog.json", "type": "application/json"},
             {"rel": "about", "href": APP, "type": "text/html", "title": "TerraPulse web app"},
             {"rel": "license", "href": f"{REPO}/blob/main/LICENSE", "type": "text/html"}]
    for s in index["sites"]:
        col = build_site(web_data / s["key"])
        links.append({"rel": "child", "href": f"{s['key']}/collection.json", "type": "application/json", "title": col["title"]})
        log(f"  stac: {s['key']} collection with {sum(1 for lk in col['links'] if lk['rel'] == 'item')} items")
    catalog = {"type": "Catalog", "stac_version": STAC_VERSION, "id": "terrapulse", "title": "TerraPulse NISAR surface change",
               "description": ("Open water, flooded vegetation, change and seasonal rhythm mapped from NASA-ISRO NISAR "
                               "L-band radar time series by the TerraPulse open-source pipeline."),
               "links": links}
    path = web_data / "catalog.json"
    path.write_text(json.dumps(catalog, indent=1))
    return path

In [ ]:
%%writefile terrapulse/__main__.py
"""Command line: python -m terrapulse <stage> [--site lower-indus]

Stages:
  search   find NISAR GCOV granules and pick one product per date
  ingest   stream each date's AOI into the common 100 m grid (resumable)
  rain     daily GPM IMERG rainfall for the site (or --site all), resumable
  process  compute products and write web layers to web/data/<site>
  stac     rebuild the STAC catalog (web/data/catalog.json) from exported layers
  all      search + ingest + process
"""
from __future__ import annotations

import argparse
import sys
from pathlib import Path

from . import catalog
from .config import SITES
from .geo import self_test
from .ingest import Grid, ingest_granule
from .remote import get_token


def main(argv=None) -> int:
    ap = argparse.ArgumentParser(prog="terrapulse")
    ap.add_argument("stage", choices=["search", "ingest", "rain", "process", "stac", "all"])
    ap.add_argument("--site", default="lower-indus", choices=sorted(SITES) + ["all"])
    ap.add_argument("--cache", default="cache", help="working directory for intermediate data")
    ap.add_argument("--web", default="web/data", help="output folder for web layers")
    ap.add_argument("--no-rain", action="store_true", help="skip rainfall (IMERG cache or NASA POWER)")
    ap.add_argument("--workers", type=int, default=8, help="parallel HTTP range requests")
    ap.add_argument("--dates", nargs="*", help="only these dates (YYYY-MM-DD)")
    ap.add_argument("--limit", type=int, help="only the first N dates")
    args = ap.parse_args(argv)

    self_test()
    sites = list(SITES.values()) if args.site == "all" else [SITES[args.site]]

    if args.stage == "stac":
        from .stac import build as build_stac
        print(f"STAC catalog: {build_stac(Path(args.web))}")
        return 0

    if args.stage == "rain":
        from .rain_imerg import fetch
        fetch(sites, get_token(), Path(args.cache))
        return 0

    status = 0
    for site in sites:
        status |= run_site(site, args)
    return status


def run_site(site, args) -> int:
    root = Path(args.cache) / site.key
    granules_path = root / "granules.json"

    if args.stage == "process":
        return process(site, root, Path(args.web) / site.key, rain=not args.no_rain)

    if args.stage == "search" or not granules_path.exists():
        found = catalog.search(site)
        chosen = catalog.select(found)
        catalog.save(chosen, granules_path)
        print(f"{site.name}: {len(found)} granules found, {len(chosen)} dates selected -> {granules_path}")
        for g in chosen:
            print(f"  {g.date}  {g.maturity:11s} {g.crid}  {g.mode} {g.pol}  core={g.core}")
        if args.stage == "search":
            return 0

    granules = catalog.load(granules_path)
    if args.dates:
        granules = [g for g in granules if g.date in set(args.dates)]
    if args.limit:
        granules = granules[: args.limit]

    token = get_token()
    grid_path = root / "grid.json"
    grid = Grid.load(grid_path) if grid_path.exists() else None
    failures = []
    for i, g in enumerate(granules, 1):
        print(f"[{i}/{len(granules)}] {g.date} {g.maturity} {g.scene}")
        try:
            if grid is None:
                from .config import GCOV_GROUP
                from .remote import open_h5
                h5, f = open_h5(g.url, token)
                epsg = int(h5[GCOV_GROUP]["projection"].attrs["epsg_code"])
                h5.close()
                f.close()
                grid = Grid.for_site(site, epsg)
                root.mkdir(parents=True, exist_ok=True)
                grid.save(grid_path)
                print(f"  grid: EPSG {grid.epsg}, {grid.width} x {grid.height} cells at {grid.res:.0f} m")
            ingest_granule(g, site, token, root / "dates", grid=grid, workers=args.workers)
        except Exception as e:  # keep going; report at the end
            print(f"  FAILED {g.date}: {type(e).__name__}: {e}")
            failures.append(g.date)
    if failures:
        print(f"Failed dates (re-run to retry): {' '.join(failures)}")
        return 1
    if args.stage == "all":
        return process(site, root, Path(args.web) / site.key, rain=not args.no_rain)
    return 0


def process(site, root: Path, out: Path, rain: bool = True) -> int:
    import warnings

    from .export import export
    from .products import compute, load_stack

    # Empty pixels outside the radar swath raise harmless all-NaN warnings.
    warnings.filterwarnings("ignore", category=RuntimeWarning)

    stack = load_stack(root)
    print(f"process {site.key}: {len(stack.dates)} dates, grid {stack.grid.width} x {stack.grid.height}")
    res = compute(stack, site)
    rain_data = None
    if rain:
        from .rain_imerg import load as load_imerg
        rain_data = load_imerg(root / "rain_imerg.json")
        if rain_data:
            print(f"  rainfall: GPM IMERG ({len(rain_data['series'].get('all', {}))} days)")
        else:
            from .rain import fetch
            try:
                rain_data = fetch(site)
            except Exception as e:
                print(f"  rainfall skipped: {e}")
    export(stack, res, site, out, rain=rain_data)
    return 0


if __name__ == "__main__":
    sys.exit(main())

## Run

In [ ]:
# Earthdata Login token (hidden input, kept only in this session's memory)
from getpass import getpass
if not os.environ.get("EARTHDATA_TOKEN"):
    os.environ["EARTHDATA_TOKEN"] = getpass("Earthdata Login token: ").strip()
print("token set:", bool(os.environ["EARTHDATA_TOKEN"]))

In [ ]:
# 1. Daily GPM IMERG rainfall for every zone of both sites (resumable)
import subprocess
if RUN_IMERG:
    r = subprocess.run([sys.executable, "-u", "-m", "terrapulse", "rain", "--site", "all"])
    if r.returncode != 0:
        print("\nRainfall step did not finish. If the message above mentions GES DISC, approve the "
              "'NASA GESDISC DATA ARCHIVE' application in Earthdata Login and run this cell again. "
              "The rest of the notebook still works and uses NASA POWER rainfall instead.")

In [ ]:
# 2 to 4. Search, stream and process each NISAR site
WORKERS = 32 if IN_COLAB else 8
for site in NISAR_SITES:
    print("=" * 30, site, "=" * 30)
    !{sys.executable} -m terrapulse search --site {site}
    !{sys.executable} -u -m terrapulse ingest --site {site} --workers {WORKERS}
    !{sys.executable} -u -m terrapulse process --site {site}

In [ ]:
# Quick look at each processed site
import json
import matplotlib.pyplot as plt
from PIL import Image

for site in NISAR_SITES:
    web = WORK / "web" / "data" / site
    man = json.loads((web / "manifest.json").read_text())
    print(site, "passes:", len(man["dates"]), " water thresholds:", man["params"]["water_thresholds_by_mode"])
    fig, ax = plt.subplots(1, 3, figsize=(18, 6))
    ax[0].imshow(Image.open(web / "products" / "water_occurrence.png"), cmap="Blues"); ax[0].set_title("Water occurrence")
    ax[1].imshow(Image.open(web / "products" / "rhythm.png").convert("RGB")); ax[1].set_title("Surface rhythm classes")
    d = man["dates"][0]["date"]
    ax[2].imshow(Image.open(web / "dates" / f"{d}_class.png").convert("RGB")); ax[2].set_title(f"Water classes {d}")
    for a in ax: a.axis("off")
    plt.show()
    z = man["zones"]["all"]["series"]
    plt.figure(figsize=(14, 3))
    plt.plot([x["date"] for x in man["dates"]], [s["water_km2"] if s else None for s in z], "o-", label="open water")
    plt.plot([x["date"] for x in man["dates"]], [s["flooded_veg_km2"] if s else None for s in z], "s--", label="flooded vegetation")
    plt.xticks(rotation=60); plt.ylabel("km2"); plt.legend(); plt.title(man["site"]["name"]); plt.show()

In [ ]:
# 5. Download one zip with everything the app needs (web layers, rainfall, small analysis cache)
import zipfile
out = WORK / "terrapulse_results.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for site in NISAR_SITES:
        for p in (WORK / "web" / "data" / site).rglob("*"):
            if p.is_file():
                z.write(p, p.relative_to(WORK).as_posix())
        for p in (WORK / "cache" / site).rglob("*"):
            if p.is_file():
                z.write(p, p.relative_to(WORK).as_posix())
    for p in (WORK / "cache").glob("*/rain_imerg.json"):
        z.write(p, p.relative_to(WORK).as_posix())
print(out.name, round(out.stat().st_size / 1e6, 1), "MB")
if IN_COLAB:
    from google.colab import files
    files.download(str(out))